# Legal Document Summary
The scrapes a legal document from a website and uses an LLM to summarize.

**Use at your own risk**

This code uses the qwen3:14b model running on the local computer. Change model as neeeded.

Only the first 20,000 characters of the document are fetched. For longer documents, this may cut-off some of the text.

In [ ]:
from bs4 import BeautifulSoup
from openai import OpenAI
from IPython.display import Markdown, display
import requests

# TODO: Update to your api and model of choice
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model="qwen3:14b"

system_prompt = """
You are a helpful legal assistant that helps to call out important points in legal documents.
Your response should include plain english and not legal terms.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.

At the end of the output, include a disclaimer that says "This summary is not legal advice, and should not be used as a substitute for consulting with a qualified attorney. Always read the full legal document and consult with a qualified attorney if you have any questions or concerns about your rights or obligations under the agreement."
"""

user_prompt = """
You will be given a legal document and your job is to output:
1. A title that summarizes the purpose of the document.
2. A one paragraph summary of the document.
3. A bulleted list that summarizes the most important points in the document that directly impact the users:
    - Rights
    - Security
    - Financial implications
    - Ability to withdraw from the agreement
    - Any other important implications for the user

Here is the legal document:

"""


# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 20,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:20_000]


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]


website = fetch_website_contents("https://openai.com/policies/")
messages = [{"role":"system", "content": system_prompt}, {"role": "user", "content": user_prompt + website}]

# Step 3: Call OpenAI
response = openai.chat.completions.create(
    model = model,
    messages = messages
)

display(Markdown(response.choices[0].message.content))